# 10 · Synthetic in-distribution and retarget results

Loads the tables under `runs/jepa-replenishment/synthetic/` and states the
success rules beside them. Those files are git-ignored. When they are
absent this notebook prints the published command that fills them and
stops. It does not plot empty numbers.

Set `JEPA_SYNTHETIC_DIR` to read a different directory. The published
log, fit, and tuning studies are in the root README section
[Latent replenishment](../README.md#latent-replenishment).


In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

from src.jepa.commands import EVAL_COMMAND

directory = Path(os.environ.get("JEPA_SYNTHETIC_DIR", "runs/jepa-replenishment/synthetic"))
TABLES_PRESENT = (
    (directory / "in_distribution.parquet").is_file()
    and (directory / "retarget.parquet").is_file()
)
if not TABLES_PRESENT:
    print(f"{directory} has no synthetic tables.")
    print("The published command that fills them is:")
    print(EVAL_COMMAND)


## In-distribution

Tuner holdout, `holding_rate` 0.01. The rule is printed with the table:
the learned arm beats the tuned anchor, and the bootstrap interval of that
paired difference excludes zero. A tie is not a pass. The table's floors are
never-order, random-cover, and least-squares-in-planner; the oracle model
in the planner is the ceiling. Order volume, service level, stockout rate,
and realised fill rate sit beside profit, with held-out rollout R² on the
same rows when the publish gate has opened.


In [ ]:
import pandas as pd

from src.jepa.eval import IN_DISTRIBUTION_RULE, in_distribution_passes

if TABLES_PRESENT:
    in_dist = pd.read_parquet(directory / "in_distribution.parquet")
    print(IN_DISTRIBUTION_RULE)
    print(in_dist.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
    print(
        "in_distribution_passes",
        in_distribution_passes(in_dist["latent"], in_dist["anchor"]),
    )
    warnings = directory / "comparison_warnings.json"
    if warnings.is_file():
        print(warnings.read_text())


## Retarget

Same holdout seeds, `holding_rate` 0.05, weights from the 0.01 fit. The
rule is printed with the table: the frozen planner beats the frozen
`OrderUpToPolicy` from the 0.01 study on paired `net_profit / initial_cash`.
The policy tuned at 0.05 is a ceiling and is not a pass bar.


In [ ]:
from src.jepa.eval import RETARGET_RULE, retarget_passes

if TABLES_PRESENT:
    retarget = pd.read_parquet(directory / "retarget.parquet")
    print(RETARGET_RULE)
    print(retarget.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
    print(
        "retarget_passes",
        retarget_passes(retarget["latent"], retarget["anchor"]),
    )
